In [1]:
import pandas as pd

url = "https://raw.githubusercontent.com/robsalgado/personal_data_science_projects/master/topic_modeling_nmf/cnn_data_4_5.csv"

df = pd.read_csv(url)

df.head()

,url,title,body,date
0,https://www.cnn.com/2020/03/23/media/japan-abe...,Japan asked the international media to change ...,"In the new system ""Canton becomes Guangzhou an...",3/24/2020
1,https://www.cnn.com/2020/03/16/perspectives/us...,The United States is still too reliant on oil,Saudi Arabia's decision to open its taps comes...,3/24/2020
2,https://www.cnn.com/2020/03/23/investing/globa...,Global stocks and US futures rise as policymak...,The promise of unlimited support for markets f...,3/24/2020
3,https://www.cnn.com/2020/03/24/economy/china-e...,China is trying to revive its economy without ...,The country where the pandemic began was almos...,3/24/2020
4,https://www.cnn.com/2020/03/24/business/bailou...,Companies that binged on buybacks now seek bai...,"Now, some of the same companies that binged on...",3/24/2020


In [2]:
print("Dataset Shape:", df.shape)

print("\nColumn Names:")
print(df.columns.tolist())

print("\nMissing Values:")
print(df.isnull().sum())

print("\nDuplicate Rows:", df.duplicated().sum())


Dataset Shape: (301, 4)

Column Names:
['url', 'title', 'body', 'date']

Missing Values:
url      0
title    0
body     0
date     0
dtype: int64

Duplicate Rows: 0


In [3]:
import re
import nltk

nltk.download('stopwords')

from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))

def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    words = text.split()
    words = [word for word in words if word not in stop_words]
    return ' '.join(words)

df['clean_body'] = df['body'].apply(clean_text)

df[['body', 'clean_body']].head()

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


,body,clean_body
0,"In the new system ""Canton becomes Guangzhou an...",new system canton becomes guangzhou tientsin b...
1,Saudi Arabia's decision to open its taps comes...,saudi arabias decision open taps comes talks o...
2,The promise of unlimited support for markets f...,promise unlimited support markets us federal r...
3,The country where the pandemic began was almos...,country pandemic began almost completely shut ...
4,"Now, some of the same companies that binged on...",companies binged buybacks line receive taxpaye...


In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    max_df=0.95,
    min_df=2,
    max_features=5000
)

X = tfidf.fit_transform(df['clean_body'])

print("TF-IDF Matrix Shape:", X.shape)

TF-IDF Matrix Shape: (301, 5000)


In [5]:
from sklearn.decomposition import NMF

n_topics = 5

nmf_model = NMF(
    n_components=n_topics,
    random_state=42,
    init='nndsvda'
)

W = nmf_model.fit_transform(X)
H = nmf_model.components_

print("W Shape:", W.shape)
print("H Shape:", H.shape)

W Shape: (301, 5)
H Shape: (5, 5000)


In [7]:
feature_names = tfidf.get_feature_names_out()

for topic_idx, topic in enumerate(H):
    top_words = [
        feature_names[i]
        for i in topic.argsort()[-10:][::-1]
    ]

    print(f"Topic {topic_idx + 1}:")
    print(", ".join(top_words))
    print()

Topic 1:
workers, said, employees, amazon, stores, company, instacart, walmart, food, customers

Topic 2:
oil, saudi, arabia, russia, prices, production, opec, energy, producers, trump

Topic 3:
economy, unemployment, stimulus, economic, billion, us, government, said, businesses, financial

Topic 4:
said, people, home, trump, work, news, new, fox, working, video

Topic 5:
airlines, airline, flights, carriers, industry, billion, passengers, travel, said, flight



In [8]:
import numpy as np

df['topic'] = W.argmax(axis=1)

df[['title', 'topic']].head(10)

,title,topic
0,Japan asked the international media to change ...,3
1,The United States is still too reliant on oil,1
2,Global stocks and US futures rise as policymak...,2
3,China is trying to revive its economy without ...,2
4,Companies that binged on buybacks now seek bai...,4
5,"Amazon hiring 100,000 new distribution workers...",0
6,These companies are hiring thousands of new em...,0
7,Cops in the toilet paper aisle: Grocery stores...,0
8,AT&T CEO on coronavirus: This is 'a time of war',3
9,Saudi Arabia just won control of the oil market,1


In [9]:
topic_names = {
    0: "Business & Retail",
    1: "Oil & Energy",
    2: "Economy & Finance",
    3: "People & Employment",
    4: "Airlines & Travel"
}

df['topic_name'] = df['topic'].map(topic_names)

df[['title', 'topic_name']].head(10)

,title,topic_name
0,Japan asked the international media to change ...,People & Employment
1,The United States is still too reliant on oil,Oil & Energy
2,Global stocks and US futures rise as policymak...,Economy & Finance
3,China is trying to revive its economy without ...,Economy & Finance
4,Companies that binged on buybacks now seek bai...,Airlines & Travel
5,"Amazon hiring 100,000 new distribution workers...",Business & Retail
6,These companies are hiring thousands of new em...,Business & Retail
7,Cops in the toilet paper aisle: Grocery stores...,Business & Retail
8,AT&T CEO on coronavirus: This is 'a time of war',People & Employment
9,Saudi Arabia just won control of the oil market,Oil & Energy


In [10]:
def predict_topic(text):
    cleaned = clean_text(text)

    text_vector = tfidf.transform([cleaned])
    topic_scores = nmf_model.transform(text_vector)

    topic_id = topic_scores.argmax()

    return topic_names[topic_id]


article = """
Saudi Arabia and other oil producing countries are discussing
oil production and energy prices in the global market.
"""

print("Predicted Topic:", predict_topic(article))

Predicted Topic: Oil & Energy


In [11]:
test_articles = [
    "Oil prices are rising as producers discuss reducing global production.",

    "Stock markets fell as investors worried about the economy and unemployment.",

    "Amazon is hiring thousands of workers for its new distribution centers.",

    "Airlines are reducing flights as passenger demand falls."
]

for i, article in enumerate(test_articles, 1):
    print(f"Article {i}:")
    print("Predicted Topic:", predict_topic(article))
    print("-" * 50)

Article 1:
Predicted Topic: Oil & Energy
--------------------------------------------------
Article 2:
Predicted Topic: Economy & Finance
--------------------------------------------------
Article 3:
Predicted Topic: Business & Retail
--------------------------------------------------
Article 4:
Predicted Topic: Airlines & Travel
--------------------------------------------------


In [12]:
import ipywidgets as widgets
from IPython.display import display, clear_output

text_box = widgets.Textarea(
    placeholder="Paste your news article here...",
    description="Article:",
    layout=widgets.Layout(width="90%", height="150px")
)

button = widgets.Button(
    description="Predict Topic",
    button_style="primary"
)

output = widgets.Output()

def on_predict(b):
    with output:
        clear_output()

        text = text_box.value.strip()

        if not text:
            print("Please enter an article first.")
            return

        topic = predict_topic(text)
        print("Predicted Topic:")
        print(topic)

button.on_click(on_predict)

display(text_box, button, output)

Textarea(value='', description='Article:', layout=Layout(height='150px', width='90%'), placeholder='Paste your…

Button(button_style='primary', description='Predict Topic', style=ButtonStyle())

Output()